In [ ]:
#Required libraries for the project
import torch
import transformers
from transformers import pipeline
from nltk.tokenize import sent_tokenize
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
import nltk
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.verification.coverage_analyzer import (
    extract_claims,
    verify_claim,
    calculate_coverage,
    print_report
)

from hallucination.evidence_mapper import (
    split_evidence,
    map_evidence
)


[nltk_data] Downloading package punkt to C:\Users\Sinjini
[nltk_data]     Laha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Sinjini
[nltk_data]     Laha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# Load the NLI model
nli = pipeline(
    "text-classification",
    model="MoritzLaurer/deberta-v3-base-mnli-fever-anli",
    device=-1
)

print("Model loaded successfully")


#Define a sample answer and evidence for testing the NLI model
sample = {
    "answer":
    "Tesla was founded in 2003. It is headquartered in Austin. Elon Musk founded Tesla.",

    "evidence":
    "Tesla was founded in 2003. Tesla is headquartered in Austin, Texas."
}


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 675.57it/s]


Model loaded successfully


In [3]:
claims = extract_claims(
    sample["answer"]
)

for claim in claims:

    mapping = map_evidence(
        claim,
        sample["evidence"],
        nli
    )

    print("="*60)

    print("CLAIM:")
    print(mapping["claim"])

    print("\nBEST EVIDENCE:")
    print(mapping["evidence"])

    print("\nLABEL:")
    print(mapping["label"])

    print("\nMATCH SCORE:")
    print(round(mapping["score"],4))

CLAIM:
Tesla was founded in 2003.

BEST EVIDENCE:
Tesla was founded in 2003.

LABEL:
entailment

MATCH SCORE:
0.9983
CLAIM:
It is headquartered in Austin.

BEST EVIDENCE:
Tesla was founded in 2003.

LABEL:
neutral

MATCH SCORE:
0.9995
CLAIM:
Elon Musk founded Tesla.

BEST EVIDENCE:
Tesla is headquartered in Austin, Texas.

LABEL:
neutral

MATCH SCORE:
0.9973


In [6]:
from src.hallucination.taxonomy_classifier import (
    TaxonomyClassifier
)

taxonomy = TaxonomyClassifier()

In [8]:
claim = "Tesla was founded in 2005."

evidence = "Tesla was founded in 2003."

print(
    taxonomy.classify(
        claim,
        evidence
    )
)
claim = "Paris is the capital of Germany."

evidence = "Paris is the capital of France."

print(
    taxonomy.classify(
        claim,
        evidence
    )
)
claim = "Bill Gates discovered penicillin."

evidence = "Alexander Fleming discovered penicillin."

print(
    taxonomy.classify(
        claim,
        evidence
    )
)

DATE_HALLUCINATION
LOCATION_HALLUCINATION
ENTITY_HALLUCINATION


In [14]:
claims = extract_claims(
    sample["answer"]
)

for claim in claims:

    mapping = map_evidence(
        claim,
        sample["evidence"],
        nli
    )

    hallucination_type = taxonomy.classify(
        claim,
        mapping["evidence"]
    )

    print("=" * 60)

    print("CLAIM:")
    print(claim)

    print("\nEVIDENCE:")
    print(mapping["evidence"])

    print("\nTYPE:")
    print(hallucination_type)

CLAIM:
Tesla was founded in 2003.

EVIDENCE:
Tesla was founded in 2003.

TYPE:
SUPPORTED_OR_UNKNOWN
CLAIM:
It is headquartered in Austin.

EVIDENCE:
Tesla is headquartered in Austin, Texas.

TYPE:
LOCATION_HALLUCINATION
CLAIM:
Elon Musk founded Tesla.

EVIDENCE:
Tesla is headquartered in Austin, Texas.

TYPE:
LOCATION_HALLUCINATION


In [11]:
for sentence in split_evidence(
    sample["evidence"]
):

    result = verify_claim(
        "It is headquartered in Austin.",
        sentence,
        nli
    )

    print("\nSentence:")
    print(sentence)

    print(result)


Sentence:
Tesla was founded in 2003.
{'label': 'neutral', 'score': 0.9994897842407227}

Sentence:
Tesla is headquartered in Austin, Texas.
{'label': 'entailment', 'score': 0.9747897982597351}


In [13]:
claims = extract_claims(sample["answer"])

for claim in claims:
    mapping = map_evidence(
        claim,
        sample["evidence"],
        nli
    )

    print(mapping)

{'claim': 'Tesla was founded in 2003.', 'evidence': 'Tesla was founded in 2003.', 'label': 'entailment', 'score': 0.9982779026031494}
{'claim': 'It is headquartered in Austin.', 'evidence': 'Tesla is headquartered in Austin, Texas.', 'label': 'entailment', 'score': 0.9747897982597351}
{'claim': 'Elon Musk founded Tesla.', 'evidence': 'Tesla is headquartered in Austin, Texas.', 'label': 'neutral', 'score': 0.9972965121269226}
